In [37]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch
import pandas as pd
import numpy as np
os.environ["HUGGINGFACE_TOKEN"] = "INSERT TOKEN"
cache_dir='  '
#model_name = "microsoft/phi-4"
#model_name="google/gemma-2-9b-it"
#model_name="meta-llama/Llama-3.2-3B-Instruct"
#model_name="meta-llama/Llama-3.1-8B-Instruct"
#model_name="CohereLabs/aya-expanse-8b"
#model_name="tiiuae/falcon-7b-instruct"
model_name="mistralai/Mixtral-8x7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto", use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)


#model_name = "tiiuae/Falcon3-Mamba-7B-Instruct"

#model = AutoModelForCausalLM.from_pretrained(
#    model_name,
#    torch_dtype=torch.bfloat16,
#    device_map="auto",
# cache_dir=cache_dir
#)
#tokenizer = AutoTokenizer.from_pretrained(model_name)


/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import numpy as np
file_path="onestopv2.csv"
df=pd.read_csv(file_path)
#df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
#df=df_shuffled
df=df.drop(["Unnamed: 0"],axis=1)
#df=df.iloc[:,:3]
df.head()

,ID,Text,Label,LLAMA_70b_vanilla,LLAMA_70b_avg,LLAMA_70b_entropy,LLAMA_8b_vanilla,LLAMA_8b_avg,LLAMA_8b_entropy,LLAMA_3b_vanilla,...,phi_entropy,AYA_32B_vanilla,AYA_32B_avg,AYA_32B_entropy,AYA_8B_vanilla,AYA_8B_avg,AYA_8B_entropy,mixtral_vanilla,mixtral_avg,mixtral_entropy
0,139,Glastonbury Festival wants to fight a war agai...,1,4,4.132964,0.033327,3,3.000000,0.089353,2,...,0.061545,3,3.0293313394467987,0.010661,3,3.090629868515556,0.027738,3,2.995240,0.005209
1,14,Astronaut Scott Kelly has just spent 340 days ...,1,4,4.853025,0.091641,4,5.000000,0.093016,4,...,0.063616,4,4.003076806791569,0.010678,4,3.73293554924021,0.047705,3,2.819735,0.046998
2,223,An extraordinary press conference at Leicester...,2,6,6.260828,0.048795,6,5.889273,0.029590,4,...,0.083819,4,4.320188002761967,0.051147,4,4.525261945681393,0.069339,6,6.022896,0.010622
3,202,"The Manchester United manager, Sir Alex Fergus...",2,6,5.924142,0.022831,6,5.915977,0.055172,4,...,0.056396,4,3.623909585070578,0.053847,4,4.250826420157068,0.050424,3,2.998994,0.002035
4,445,Robert Mysłajek stops dead. Between two paw pr...,3,7,6.697060,0.052146,4,4.853025,0.091641,4,...,0.093791,4,3.9981597518398075,0.012330,4,4.352445047250399,0.056907,6,6.118490,0.035705


In [38]:
## Gets readability scores without eliciting model confidence

## MODELS: LLAMA 3B or 8B or 70B or Mixtral or Aya 8B or 32B or phi 4 or gemma 2b/9b/27B
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,1]
    score=df.iloc[i,2]

    prompt=f"Rate the readability of these news articles with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"        
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM')
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['mixtral_vanilla']=scores_llm_vanilla
df['mixtral_avg']=scores_llm_avg
df['mixtral_entropy']=entropies
#print(corrs)
df.to_csv("onestopv2.csv")

ERROR - FIRST TOKEN NOT NUM
ERROR - FIRST TOKEN NOT NUM
0.2666196693651566 565 0.31613092710171337 565


In [19]:
## Gets readability scores and confidence scores

corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,1]
    score=df.iloc[i,2]

    prompt=f"Rate the readability of the text with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=10,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    logits_last=scores[-3][0]
    logits_fifth_to_last = scores[-7][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM')
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
        
    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-2][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-1][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                
                print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                flag2=True
                verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_phi_vanilla']=scores_llm_vanilla
df['conf_phi_avg']=scores_llm_avg
df['conf_phi_entropy']=entropies
df['conf_phi_verbal_conf']=verbal_confs
#print(corrs)
df.to_csv("onestopv2.csv")

0.4647144311190153 567 0.4968917386558468 567
-0.6808680579450822


In [23]:
##FALCON 
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,1]
    score=df.iloc[i,2]

    prompt=f"All of the news articles were rewritten so that they would be suitable for people who are learning the English language. Rate the readability of these news articles between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"        
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    outputs = model.generate(**model_inputs, max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['Falcon_vanilla']=scores_llm_vanilla
df['Falcon_avg']=scores_llm_avg
df['Falcon_entropy']=entropies
#print(corrs)
df.to_csv("onestopv2.csv")

ERROR - FIRST TOKEN NOT NUM ['RE', 'UR', 'PE', 'OR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'OR', 'PE', 'UR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'UR', 'RES', 'OR', 'LE']
ERROR - FIRST TOKEN NOT NUM ['IST', 'OC', 'IGN', '.', 'Ġ', ')']
ERROR - FIRST TOKEN NOT NUM ['ĠWhat', 'Ġ', 'ĠThe', 'ĠHow', 'ĠIn', 'ĠWhich']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'OR', 'UR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['IST', 'OC', 'IGN', 'ESS', 'UMP', '.']
ERROR - FIRST TOKEN NOT NUM ['RE', 'OR', 'PE', 'UR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'UR', 'OR', 'RES', 'PE', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'UR', 'PE', 'OR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'UR', 'LE', 'OR', 'RES']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'OR', 'UR', 'RES', 'AR']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'OR', 'UR', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'OR', 'UR', 'PE', 'RES', 'LE']
ERROR - FIRST TOKEN NOT NUM ['RE', 'PE', 'OR', 'UR', 'RES', 'LE']


ValueError: not enough values to unpack (expected 3, got 0)